# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Dataset Exploration with `mlcroissant`

This notebook provides step-by-step guidance for loading, exploring, and performing exploratory data analysis (EDA) on the FAIR² dataset package using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema, accessible from the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Install mlcroissant (if not already installed)
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`. This section demonstrates how to instantiate a Dataset object, access its metadata, and get an overview of the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata with mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Single object (not dict/list)

# Display dataset metadata overview
print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Date Published: {metadata.datePublished}")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Explore the record sets (`@id`s), fields, and columns available in the dataset according to the Croissant schema.

Use the `@id` for referencing record sets, fields, and columns in all subsequent steps (as per Croissant best practice).

In [ ]:
# List available record sets and their fields by @id
print("Available Record Sets:")
for record_set in dataset.record_sets:
    print(f"  RecordSet @id: {record_set['@id']}")
    fields = record_set.get('field', [])
    # Croissant YAML can specify a single field as dict, not list
    if isinstance(fields, dict):
        fields = [fields]
    print("    Fields:")
    for field in fields:
        if isinstance(field, dict):
            print(f"      Field @id: {field.get('@id', '<no id>')}, name: {field.get('name', '<no name>')}")
        elif isinstance(field, str):
            print(f"      Field @id: {field}")
    print("")

:::{admonition} **Tip**
The record set `@id` and each field's `@id` must be used to extract data below.
:::

Next, let's preview a few records from the main record set. For clinical datasets, there is usually a main patient/case table.


In [ ]:
# Identify expected main record set (usually the only one, or named like 'cases', 'patients', etc)
# For this dataset, let's programmatically select the first record set:
main_record_set_id = None
if len(dataset.record_sets):
    main_record_set_id = dataset.record_sets[0]['@id']

print(f"Previewing records from RecordSet @id: {main_record_set_id}\n")
for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
    if i >= 3:
        break
    print(rec)


## 3. Data Extraction
Load all tabular records from the record set(s) into pandas DataFrames for analysis. Use the record set's and fields' `@id`s from the previous overview.


In [ ]:
# Extract all record set @ids for batch DataFrame loading
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df

# Show DataFrame columns for the main record set
if main_record_set_id:
    print(f"Columns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
This section demonstrates common data processing tasks: filtering on a numeric field, normalization, and optionally grouping.

We use only `@id` to dynamically retrieve columns. For demonstration, let's pick `age` if such a field/column exists (variable names may vary).

In [ ]:
# Pick a numeric field @id from the DataFrame columns
# We attempt to find an 'age' field or substitute with another numeric

df = dataframes[main_record_set_id]

import numpy as np

# Helper: find first numeric column
possible_numeric = [col for col in df.columns if 'age' in col.lower()]
if possible_numeric and pd.api.types.is_numeric_dtype(df[possible_numeric[0]]):
    numeric_field_id = possible_numeric[0]
else:
    # fallback: first integer/float column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
        else:
            numeric_field_id = None

if not numeric_field_id:
    print("No numeric field found - skipping numeric EDA example.")
else:
    print(f"Numeric field selected (@id): {numeric_field_id}")

    threshold = df[numeric_field_id].quantile(0.25)  # use 25th percentile as example threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std(ddof=0)
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to find a groupable field (e.g. sex, anatomical_location, diagnosis, etc)
    candidates = [col for col in df.columns if any(token in col.lower() for token in ["sex", "site", "anatom", "diagnosis", "comorb", "status"])]
    group_field = candidates[0] if candidates else None

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field}:")
        display(grouped_df)
    else:
        print("No suitable group field found for grouping.")


## 5. Visualization
Visualize distributions or relationships between fields in the dataset. Here we plot the distribution of the numeric field and, if available, compare by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field_id' in locals() and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field found to plot.")

## 6. Conclusion

- We loaded and explored the FAiR² dataset using the `mlcroissant` library, referencing all entities by their `@id` fields for consistency and reproducibility.
- Using Croissant, we programmatically listed record sets and fields, loaded tabular records into DataFrames, and performed basic EDA including filtering, normalization, grouping, and visualization.
- This approach provides a robust, schema-driven exploration of complex, FAIR datasets for clinical or biomedical applications.
